# Section 3: Robustness, Where You Go Beyond (28 pts)

ECON 5200 Midterm Project · Replication Study

**AI is allowed in this notebook**, as a documented co-pilot under P.R.I.M.E. File every exchange, in full, in [`ai-appendix.md`](../ai-appendix.md) under the exercise it belongs to. If you cannot explain a line, you cannot submit it.

At least four exercises the original did not run. Choose from modern estimators, a specification curve, sensitivity to unobserved confounding, inference (clustering at the level of treatment assignment, a wild cluster bootstrap if clusters are few), bandwidth or instrument strength, and placebo tests, or justify an alternative. For each one: what you did, what you found, and what it implies about the original claim.

## Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from linearmodels.iv import IV2SLS

# The repository root, whether this runs from notebooks/ (Jupyter) or from the root (reproduce.py).
ROOT = Path.cwd() if (Path.cwd() / "download_data.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
(ROOT / "output").mkdir(exist_ok=True)

from download_data import MY_PAPER, PAPERS, fetch

print(f"Setup complete. Paper: {MY_PAPER}")

## Data

In [ ]:
paths = fetch(MY_PAPER)
raw = pd.read_stata(paths[0])

df = raw.copy()
df["agesq"]   = df["age"] ** 2
df["expersq"] = df["exper"] ** 2

NEEDED = ["lwage", "educ", "exper", "expersq", "age", "agesq", "nearc4",
          "black", "south", "smsa", "smsa66"] + [f"reg66{i}" for i in range(1, 9)]
df = df.dropna(subset=NEEDED).copy()
assert len(df) == 3010, f"N = {len(df)}, expected 3,010"

print("exper == age − educ − 6:", (df["exper"] == df["age"] - df["educ"] - 6).mean().round(4))
display(df[["lwage", "educ"]].agg(["mean", "std"]).T)


## Specification

In [ ]:
EXOG     = "black + south + smsa + smsa66 + " + " + ".join(f"reg66{i}" for i in range(1, 9))
CONTROLS = f"exper + expersq + {EXOG}"

# Panel A -> experience exogenous
ols_fit = smf.ols(f"lwage ~ educ + {CONTROLS}", data=df).fit()
fs_fit  = smf.ols(f"educ ~ nearc4 + {CONTROLS}", data=df).fit()
rf_fit  = smf.ols(f"lwage ~ nearc4 + {CONTROLS}", data=df).fit()
iv_fit  = IV2SLS.from_formula(f"lwage ~ 1 + {CONTROLS} + [educ ~ nearc4]",
                              data=df).fit(cov_type="unadjusted", debiased=True)

# Panel B -> experience endogenous | instrumented by age and age²
fsB_fit = smf.ols(f"educ ~ nearc4 + age + agesq + {EXOG}", data=df).fit()
rfB_fit = smf.ols(f"lwage ~ nearc4 + age + agesq + {EXOG}", data=df).fit()
ivB_fit = IV2SLS.from_formula(
    f"lwage ~ 1 + {EXOG} + [educ + exper + expersq ~ nearc4 + age + agesq]",
    data=df).fit(cov_type="unadjusted", debiased=True)

## The four exercises

### Exercise 1: Instrument strength

**What I did.** I adapted the starter’s first_stage function to regress educ on nearc4 and the 14 controls by OLS, dropping any control that is constant in the sample. I recorded the nearc4 coefficient, its SE, and the first-stage F (squared t-stat) with both classical and HC1 robust SEs.

In [ ]:
OUT = Path("../output")  # adjust if your template sets this differently
OUT.mkdir(exist_ok=True)

CTRL_LIST = ["exper", "expersq", "black", "south", "smsa", "smsa66"] + [f"reg66{i}" for i in range(1, 9)]
assert CTRL_LIST == [c.strip() for c in CONTROLS.split("+")]  # must match the CONTROLS string

In [ ]:
# Adapted from the course starter (reference/replication_starter_card1995.ipynb):
# added SE, default data=df, and dropping of controls that are constant in a subsample
def first_stage(controls, instrument="nearc4", data=df):
    dropped = [c for c in controls if data[c].nunique() == 1]
    controls = [c for c in controls if c not in dropped]
    X = sm.add_constant(data[[instrument] + controls])
    classical = sm.OLS(data["educ"], X).fit()
    robust = sm.OLS(data["educ"], X).fit(cov_type="HC1")
    return {"coef": classical.params[instrument], "se": classical.bse[instrument],
            "F": classical.tvalues[instrument] ** 2, "F_robust": robust.tvalues[instrument] ** 2,
            "n": int(classical.nobs), "dropped": dropped}

fs = first_stage(CTRL_LIST)
print(f"First stage on nearc4: {fs['coef']:+.3f} (SE {fs['se']:.3f}), "
      f"F = {fs['F']:.1f} (robust {fs['F_robust']:.1f}), n = {fs['n']:,}, dropped: {fs['dropped']}")

**What I found.** The nearc4 coefficient and SE, and where F sits relative to 10 and 104.7. Also compare F with F_robust and confirm n = 3,010 with nothing dropped.

**What it implies about the original claim.** Living near a four-year college raises schooling by about 0.32 years, so nearc4 is a relevant instrument. With F = 13.3 (robust 14.1), it passes the old F > 10 rule but falls well short of Lee et al.’s 104.7, so the first stage is only moderately strong.

### Exercise 2: Anderson-Rubin

**What I did.** I adapted the starter’s anderson_rubin function to test each candidate return to schooling on a grid, keeping values where nearc4 is insignificant at 5% in the reduced form, with a check for gaps and a wider grid if the set hits the edge. I ran it on the full sample with nonrobust and HC1 SEs and compared it with the 2SLS Wald interval.

In [6]:
# Adapted from the course starter (reference/replication_starter_card1995.ipynb):
# added data=df default, cov_type option, constant-control dropping and a gap check
def anderson_rubin(controls, instrument="nearc4", grid=np.linspace(-0.2, 0.6, 801), data=df, cov_type="HC1"):
    controls = [c for c in controls if data[c].nunique() > 1]
    X = sm.add_constant(data[[instrument] + controls])
    keep = [i for i, b0 in enumerate(grid)
            if sm.OLS(data["lwage"] - b0 * data["educ"], X).fit(cov_type=cov_type).pvalues[instrument] > 0.05]
    if not keep:
        return None
    if keep[-1] - keep[0] + 1 != len(keep):
        print("Warning: AR set is not one interval (has a gap)")
    edge = keep[0] == 0 or keep[-1] == len(grid) - 1
    return grid[keep[0]], grid[keep[-1]], edge

WIDE = np.linspace(-1, 2, 3001)

def ar_with_widen(controls, data=df, cov_type="HC1"):
    ar = anderson_rubin(controls, data=data, cov_type=cov_type)
    if ar is not None and ar[2]:
        ar = anderson_rubin(controls, grid=WIDE, data=data, cov_type=cov_type)
    return ar

for ct in ["nonrobust", "HC1"]:
    lo, hi, edge = ar_with_widen(CTRL_LIST, cov_type=ct)
    print(f"AR ({ct}): [{lo:.3f}, {hi:.3f}], touches edge: {edge}")
wald = iv_fit.conf_int().loc["educ"]
print(f"Wald: [{wald.iloc[0]:.3f}, {wald.iloc[1]:.3f}]")

AR (nonrobust): [0.025, 0.284], touches edge: False
AR (HC1): [0.029, 0.281], touches edge: False
Wald: [0.024, 0.239]


**What I found.** The AR intervals are [0.025, 0.284] (nonrobust) and [0.029, 0.281] (HC1), both bounded and single intervals, against a Wald interval of [0.024, 0.239]. The lower bounds are nearly identical, but the AR upper bound is about 0.04 higher.

**What it implies about the original claim.** The return to schooling stays positive and significant even under weak-instrument-robust inference, since both AR intervals exclude zero. The Wald interval understates uncertainty at the upper end, while robust and nonrobust AR give nearly the same answer.

### Exercise 3: My own sepcification

**What I did.** I wrote an iv_spec helper that runs 2SLS of lwage on educ with given controls and instruments, recording the coefficient, SE, CI, n, and joint first-stage F. I ran the baseline (a), the baseline on the family-background sample (b0), and that sample with father’s and mother’s education and momdad14 added (b), separating the sample effect from the controls effect.

In [7]:
def iv_spec(controls, instruments=["nearc4"], data=df):
    data = data.dropna(subset=controls + instruments)
    controls = [c for c in controls if data[c].nunique() > 1]
    exog = sm.add_constant(data[controls])
    fit = IV2SLS(data["lwage"], exog, data["educ"], data[instruments]).fit(cov_type="unadjusted", debiased=True)
    fs = sm.OLS(data["educ"], sm.add_constant(data[instruments + controls])).fit()
    fs_F = fs.f_test(", ".join(f"{z} = 0" for z in instruments)).fvalue  # joint F, equals t² with one instrument
    ci = fit.conf_int().loc["educ"]
    row = {"instrument": "+".join(instruments), "n": int(fit.nobs), "iv_coef": fit.params["educ"],
           "iv_se": fit.std_errors["educ"], "ci_low": ci.iloc[0], "ci_high": ci.iloc[1], "fs_F": float(fs_F)}
    return row, fit

rows = []
def add(spec, controls, instruments=["nearc4"], data=df):
    row, fit = iv_spec(controls, instruments, data)
    rows.append({"spec": spec, **row})
    return fit

FAMILY = ["fatheduc", "motheduc", "momdad14"]
fam_df = df.dropna(subset=FAMILY)

add("a) Baseline", CTRL_LIST)
add("b0) Baseline, family sample", CTRL_LIST, data=fam_df)
add("b) + family background", CTRL_LIST + FAMILY, data=fam_df)

print("Rows lost to family background:", len(df) - len(fam_df))
print(pd.DataFrame(rows).round({"iv_coef": 4, "iv_se": 4, "ci_low": 3, "ci_high": 3, "fs_F": 2}))

Rows lost to family background: 790
                          spec instrument     n  iv_coef   iv_se  ci_low  \
0                  a) Baseline     nearc4  3010   0.1315  0.0550   0.024   
1  b0) Baseline, family sample     nearc4  2220   0.0799  0.0744  -0.066   
2       b) + family background     nearc4  2220   0.0829  0.0768  -0.068   

   ci_high   fs_F  
0    0.239  13.26  
1    0.226   6.62  
2    0.234   6.95  


**What I found.** Restricting to the 2,220 people with family background data lowers the estimate from 0.1315 (SE 0.0550) to 0.0799 (SE 0.0744) and the first-stage F from 13.3 to 6.6. Adding the family controls barely changes this (0.0829, SE 0.0768, F 6.95), and both family-sample intervals include zero.

**What it implies about the original claim.** The drop in the estimate comes from the smaller sample, not from the family controls, so family background does not appear to confound the proximity instrument. On the family sample the instrument is weak (F < 10), so these estimates are too imprecise to distinguish from zero and may be biased toward OLS.

### Exercise 4: exclusion restriction / placebo

**What I did.** I regressed standardized IQ and KWW scores on nearc4 and the baseline controls with HC1 SEs, as placebo tests of whether college proximity predicts pre-college ability. I ran each with age controls (the clean version) and with experience controls, since experience depends on schooling.

In [8]:
PLACEBO_CTRL = {"age controls": ["age", "agesq"] + CTRL_LIST[2:],  # CTRL_LIST[2:] = black ... reg668
                "exper controls": CTRL_LIST}

placebo = []
for y in ["IQ", "KWW"]:
    d = df.dropna(subset=[y]).copy()
    d["z"] = (d[y] - d[y].mean()) / d[y].std()
    for label, ctrl in PLACEBO_CTRL.items():
        fit = sm.OLS(d["z"], sm.add_constant(d[["nearc4"] + ctrl])).fit(cov_type="HC1")
        placebo.append({"outcome": y, "controls": label, "coef": fit.params["nearc4"],
                        "se": fit.bse["nearc4"], "p": fit.pvalues["nearc4"], "n": int(fit.nobs)})

placebo = pd.DataFrame(placebo)
placebo.to_csv(OUT / "ex4_placebo.csv", index=False)
print(placebo.round({"coef": 3, "se": 3, "p": 4}))

  outcome        controls   coef     se       p     n
0      IQ    age controls  0.059  0.048  0.2186  2061
1      IQ  exper controls  0.044  0.044  0.3214  2061
2     KWW    age controls  0.088  0.037  0.0188  2963
3     KWW  exper controls  0.111  0.040  0.0051  2963


**What I found.** Proximity has no significant association with IQ (0.059 SD, p = 0.22 with age controls; 0.044 SD, p = 0.32 with experience controls; n = 2,061). It is positively associated with KWW (0.088 SD, p = 0.019 with age controls; 0.111 SD, p = 0.005 with experience controls; n = 2,963).

**What it implies about the original claim.** The IQ placebo passes, but the KWW result suggests men near colleges had somewhat higher ability or better schooling before college, which casts doubt on the exclusion restriction and could bias the IV estimate upward. Since some men were already in college by 1966, part of the KWW gap may reflect schooling itself, so this is a warning sign rather than proof that exclusion fails.